# 🏥 Amazon Comprehend Medical & Transcribe Medical Lab
### Fully Automated — SageMaker Ready
> **Fixed Version** — Handles empty transcript, audio codec issues, and auto-fallback

## ⚙️ Step 0 — Install Dependencies & Setup

In [ ]:
# Install required packages
import subprocess, sys

packages = ['boto3', 'pydub']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

# ffmpeg for audio conversion
ret = subprocess.run(['which', 'ffmpeg'], capture_output=True)
if ret.returncode != 0:
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
    print('✅ ffmpeg installed')
else:
    print('✅ ffmpeg already available')

import boto3, json, os, time, glob, uuid, wave, struct, math

# AWS clients
session   = boto3.session.Session()
REGION    = session.region_name or 'us-east-1'
ACCOUNT   = boto3.client('sts').get_caller_identity()['Account']
s3        = boto3.client('s3', region_name=REGION)
transcribe = boto3.client('transcribe', region_name=REGION)
comprehend_medical = boto3.client('comprehendmedical', region_name=REGION)

print(f'✅ Region  : {REGION}')
print(f'✅ Account : {ACCOUNT}')

## 🔐 Step 1 — Fix IAM Permissions (Run Once)

In [ ]:
iam = boto3.client('iam')
sts = boto3.client('sts')

role_arn  = sts.get_caller_identity()['Arn']
role_name = role_arn.split('/')[-2]  # assumed-role/ROLE_NAME/session
print(f'Role: {role_name}')

POLICIES = [
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
    'arn:aws:iam::aws:policy/AmazonTranscribeFullAccess',
    'arn:aws:iam::aws:policy/ComprehendMedicalFullAccess',
    'arn:aws:iam::aws:policy/IAMFullAccess',
]

for p in POLICIES:
    name = p.split('/')[-1]
    try:
        iam.attach_role_policy(RoleName=role_name, PolicyArn=p)
        print(f'✅ Attached  : {name}')
    except iam.exceptions.EntityAlreadyExistsException:
        print(f'ℹ️  Already  : {name}')
    except Exception as e:
        print(f'⚠️  {name}: {e}')

print('\n✅ Permissions ready!')

## 🪣 Step 2 — Create S3 Buckets

In [ ]:
UID = uuid.uuid4().hex[:8]

BUCKET_AUDIO_INPUT   = f'medical-audio-input-{UID}'
BUCKET_TRANSCRIBE_OUT = f'medical-transcribe-out-{UID}'
BUCKET_COMPREHEND_IN  = f'medical-comprehend-in-{UID}'
BUCKET_COMPREHEND_OUT = f'medical-comprehend-out-{UID}'

BUCKETS = [BUCKET_AUDIO_INPUT, BUCKET_TRANSCRIBE_OUT, BUCKET_COMPREHEND_IN, BUCKET_COMPREHEND_OUT]

for bucket in BUCKETS:
    try:
        if REGION == 'us-east-1':
            s3.create_bucket(Bucket=bucket)
        else:
            s3.create_bucket(
                Bucket=bucket,
                CreateBucketConfiguration={'LocationConstraint': REGION}
            )
        print(f'✅ Created: {bucket}')
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f'ℹ️  Exists : {bucket}')

print('\n✅ All buckets ready!')

## 🎙️ Step 3 — Audio File Detection + Validation + Upload

In [ ]:
import subprocess

SUPPORTED = ['mp4', 'mp3', 'wav', 'm4a', 'flac', 'ogg', 'amr', 'webm']
FORMAT_MAP = {'mp4':'mp4','mp3':'mp3','wav':'wav','m4a':'mp4',
              'flac':'flac','ogg':'ogg','amr':'amr','webm':'webm'}

# ── Search for audio files ────────────────────────────────────────────────
search_paths = ['.', '/home/sagemaker-user', os.path.expanduser('~')]
found_files = []
for path in search_paths:
    for fmt in SUPPORTED:
        found_files.extend(glob.glob(f'{path}/*.{fmt}'))
found_files = list(set(found_files))

if not found_files:
    print('❌ No audio file found in SageMaker directory.')
    print('   Upload your audio file using File Browser (left panel → Upload icon)')
    raise FileNotFoundError('Upload an audio file and re-run this cell.')

AUDIO_FILE = found_files[0]
file_ext   = AUDIO_FILE.rsplit('.', 1)[-1].lower()
MEDIA_FORMAT = FORMAT_MAP.get(file_ext, 'mp4')
size_kb = os.path.getsize(AUDIO_FILE) / 1024

print(f'✅ Audio file  : {AUDIO_FILE}')
print(f'   Format      : {MEDIA_FORMAT}')
print(f'   Size        : {size_kb:.1f} KB')

# ── Audio Diagnostics using ffprobe ──────────────────────────────────────
print('\n🔍 Running audio diagnostics...')
try:
    result = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json',
         '-show_streams', '-show_format', AUDIO_FILE],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        probe = json.loads(result.stdout)
        streams = probe.get('streams', [])
        fmt_info = probe.get('format', {})
        duration = float(fmt_info.get('duration', 0))
        print(f'   Duration    : {duration:.1f} seconds')
        for s in streams:
            if s.get('codec_type') == 'audio':
                codec = s.get('codec_name', 'unknown')
                sr = s.get('sample_rate', 'unknown')
                ch = s.get('channels', 'unknown')
                print(f'   Audio codec : {codec}')
                print(f'   Sample rate : {sr} Hz')
                print(f'   Channels    : {ch}')
        
        if duration < 1.0:
            print('\n⚠️  WARNING: Audio duration < 1 second — Transcribe may return empty!')
        elif duration > 14400:
            print('\n⚠️  WARNING: Audio > 4 hours — may hit Transcribe limits')
        else:
            print(f'   ✅ Duration looks good ({duration:.0f}s)')
    else:
        print('   ⚠️  ffprobe could not read file — proceeding anyway')
except Exception as e:
    print(f'   ⚠️  Diagnostics skipped: {e}')

# ── Convert to PCM WAV for best Transcribe compatibility ─────────────────
print('\n🔄 Converting audio to PCM WAV (best Transcribe compatibility)...')
CONVERTED_AUDIO = 'converted_audio.wav'
try:
    conv = subprocess.run(
        ['ffmpeg', '-y', '-i', AUDIO_FILE,
         '-ar', '16000',   # 16kHz sample rate (Transcribe Medical optimal)
         '-ac', '1',       # mono channel
         '-acodec', 'pcm_s16le',  # PCM 16-bit little-endian
         CONVERTED_AUDIO],
        capture_output=True, text=True, timeout=120
    )
    if conv.returncode == 0 and os.path.exists(CONVERTED_AUDIO):
        conv_size = os.path.getsize(CONVERTED_AUDIO) / 1024
        print(f'✅ Converted : {CONVERTED_AUDIO} ({conv_size:.1f} KB)')
        UPLOAD_FILE   = CONVERTED_AUDIO
        MEDIA_FORMAT  = 'wav'
    else:
        print('⚠️  Conversion failed — using original file')
        print(f'   ffmpeg stderr: {conv.stderr[-300:]}')
        UPLOAD_FILE  = AUDIO_FILE
except Exception as e:
    print(f'⚠️  Conversion error: {e} — using original file')
    UPLOAD_FILE  = AUDIO_FILE

# ── Upload to S3 ──────────────────────────────────────────────────────────
S3_AUDIO_KEY = f'audio/{os.path.basename(UPLOAD_FILE)}'
s3.upload_file(UPLOAD_FILE, BUCKET_AUDIO_INPUT, S3_AUDIO_KEY)
S3_AUDIO_URI = f's3://{BUCKET_AUDIO_INPUT}/{S3_AUDIO_KEY}'
print(f'\n✅ Uploaded to: {S3_AUDIO_URI}')

## 📝 Step 4 — Amazon Transcribe Medical Job

In [ ]:
JOB_NAME = f'MedTranscribe-{uuid.uuid4().hex[:8]}'

transcribe.start_medical_transcription_job(
    MedicalTranscriptionJobName = JOB_NAME,
    LanguageCode  = 'en-US',
    MediaFormat   = MEDIA_FORMAT,
    Media         = {'MediaFileUri': S3_AUDIO_URI},
    OutputBucketName = BUCKET_TRANSCRIBE_OUT,
    OutputKey        = 'transcripts/',
    Specialty        = 'PRIMARYCARE',
    Type             = 'DICTATION',
)

print(f'✅ Transcription job started: {JOB_NAME}')
print('   Waiting for completion...')

# Poll until done
while True:
    resp   = transcribe.get_medical_transcription_job(MedicalTranscriptionJobName=JOB_NAME)
    status = resp['MedicalTranscriptionJob']['TranscriptionJobStatus']
    ts     = time.strftime('%H:%M:%S')
    print(f'   [{ts}] Status: {status}')
    if status in ('COMPLETED', 'FAILED'):
        break
    time.sleep(15)

if status == 'FAILED':
    reason = resp['MedicalTranscriptionJob'].get('FailureReason', 'Unknown')
    print(f'\n❌ Job failed: {reason}')
else:
    TRANSCRIPT_URI = resp['MedicalTranscriptionJob']['Transcript']['TranscriptFileUri']
    print(f'\n✅ Completed! Output URI: {TRANSCRIPT_URI}')

## 📥 Step 5 — Download & Extract Transcript (with Smart Fallback)

In [ ]:
# ── Download transcript JSON from S3 ─────────────────────────────────────
# List objects to find the transcript file
resp = s3.list_objects_v2(Bucket=BUCKET_TRANSCRIBE_OUT, Prefix='transcripts/')
transcript_key = None
for obj in resp.get('Contents', []):
    if obj['Key'].endswith('.json'):
        transcript_key = obj['Key']
        break

if transcript_key:
    s3.download_file(BUCKET_TRANSCRIBE_OUT, transcript_key, 'transcription_output.json')
    print(f'✅ Downloaded: {transcript_key}')
    
    with open('transcription_output.json', 'r') as f:
        data = json.load(f)
    
    transcript_text = data.get('results', {}).get('transcripts', [{}])[0].get('transcript', '')
    print(f'   Transcript length: {len(transcript_text)} characters')
else:
    print('⚠️  No transcript JSON found in S3')
    transcript_text = ''

# ── Smart Fallback ────────────────────────────────────────────────────────
FALLBACK_USED = False
if len(transcript_text.strip()) < 50:
    print()
    print('⚠️  Transcript is empty or too short.')
    print('   Possible reasons:')
    print('   1. Audio has no clear speech (background noise, music)')
    print('   2. Audio codec not fully supported by Transcribe Medical')
    print('   3. Speaker too far from mic / very low volume')
    print()
    print('🔄 Using sample clinical transcript for Comprehend Medical demo...')
    
    transcript_text = """Patient is a 58-year-old female with a history of hypertension and 
type 2 diabetes mellitus. She presents today with complaints of chest pain radiating to 
the left arm, shortness of breath, and diaphoresis for the past 3 hours. 
She has been taking Metformin 500 mg twice daily and Lisinopril 10 mg once daily. 
She reports a family history of coronary artery disease. 
On examination, blood pressure is 160/95 mmHg, heart rate 98 beats per minute, 
oxygen saturation 94 percent on room air. ECG shows ST-segment elevation in leads II, 
III and aVF. Troponin levels are elevated at 2.3. 
Assessment: Acute inferior STEMI. 
Plan: Aspirin 325 mg stat, Nitroglycerin 0.4 mg sublingual, 
activate cardiac catheterization lab, consult cardiology, 
IV heparin bolus, NPO, continuous cardiac monitoring."""
    FALLBACK_USED = True
    print(f'✅ Sample transcript loaded ({len(transcript_text)} chars)')
else:
    print(f'\n✅ Real transcript extracted ({len(transcript_text)} chars)')

print('\n' + '='*60)
print('TRANSCRIPT TEXT:')
print('='*60)
print(transcript_text)

# Save transcript to file
with open('transcript.txt', 'w') as f:
    f.write(transcript_text)

# Upload transcript to Comprehend input bucket
s3.upload_file('transcript.txt', BUCKET_COMPREHEND_IN, 'input/transcript.txt')
print(f'\n✅ Transcript uploaded to s3://{BUCKET_COMPREHEND_IN}/input/transcript.txt')

if FALLBACK_USED:
    print('\n📌 NOTE: Fallback sample text used — to use real audio transcript:')
    print('   • Ensure audio has clear English speech')
    print('   • Try re-recording at 16kHz mono WAV')
    print('   • Check: ffprobe your_file.mp4 for codec details')

## 🔬 Step 6 — Amazon Comprehend Medical (Real-Time)

In [ ]:
print('🔬 Running Amazon Comprehend Medical — detect_entities_v2...')
print('='*60)

response = comprehend_medical.detect_entities_v2(Text=transcript_text)

entities = response.get('Entities', [])
print(f'✅ Total entities detected: {len(entities)}\n')

# Group by category
from collections import defaultdict
grouped = defaultdict(list)
for e in entities:
    grouped[e['Category']].append(e)

CATEGORY_ICONS = {
    'MEDICATION'              : '💊',
    'MEDICAL_CONDITION'       : '🩺',
    'ANATOMY'                 : '🫀',
    'TEST_TREATMENT_PROCEDURE': '🔬',
    'TIME_EXPRESSION'         : '🕐',
    'PROTECTED_HEALTH_INFORMATION': '🔒',
}

for category, items in sorted(grouped.items()):
    icon = CATEGORY_ICONS.get(category, '📋')
    print(f'{icon} {category} ({len(items)} entities)')
    print('-' * 50)
    for e in items:
        score = e['Score']
        text  = e['Text']
        etype = e.get('Type', '')
        attrs = e.get('Attributes', [])
        print(f'   • {text:<35} [{etype}]  conf: {score:.2f}')
        for attr in attrs:
            print(f'     └─ {attr["Type"]}: {attr["Text"]} (conf: {attr["Score"]:.2f})')
    print()

## 💊 Step 7 — Detect Medication Details (RxNorm)

In [ ]:
print('💊 Detect Medications with RxNorm Codes...')
print('='*60)

rx_resp = comprehend_medical.infer_rx_norm(Text=transcript_text)
rx_entities = rx_resp.get('Entities', [])

if rx_entities:
    for e in rx_entities:
        med_name = e['Text']
        score    = e['Score']
        concepts = e.get('RxNormConcepts', [])
        print(f'\n💊 Medication : {med_name}  (conf: {score:.2f})')
        for c in concepts[:3]:  # Top 3 RxNorm matches
            print(f'   RxNorm Code : {c.get("Code", "N/A")}')
            print(f'   Description : {c.get("Description", "N/A")}')
            print(f'   Score       : {c.get("Score", 0):.2f}')
        attrs = e.get('Attributes', [])
        for a in attrs:
            print(f'   {a["Type"]:<15}: {a["Text"]}')
else:
    print('ℹ️  No RxNorm medication entities found in this text.')

print(f'\n✅ RxNorm detection complete. {len(rx_entities)} medications found.')

## 🧬 Step 8 — Detect ICD-10-CM Codes (Diagnoses)

In [ ]:
print('🧬 Detect Medical Conditions with ICD-10-CM Codes...')
print('='*60)

icd_resp = comprehend_medical.infer_icd10_cm(Text=transcript_text)
icd_entities = icd_resp.get('Entities', [])

if icd_entities:
    for e in icd_entities:
        condition = e['Text']
        score     = e['Score']
        concepts  = e.get('ICD10CMConcepts', [])
        print(f'\n🩺 Condition : {condition}  (conf: {score:.2f})')
        for c in concepts[:3]:
            print(f'   ICD-10 Code : {c.get("Code", "N/A")}')
            print(f'   Description : {c.get("Description", "N/A")}')
            print(f'   Score       : {c.get("Score", 0):.2f}')
else:
    print('ℹ️  No ICD-10-CM entities found in this text.')

print(f'\n✅ ICD-10-CM detection complete. {len(icd_entities)} conditions found.')

## 📊 Step 9 — Summary Report

In [ ]:
print('='*70)
print('              CLINICAL NLP ANALYSIS — SUMMARY REPORT')
print('='*70)

source = '⚠️  Sample fallback text' if FALLBACK_USED else '✅ Real audio transcription'
print(f'\nTranscript Source : {source}')
print(f'Transcript Length : {len(transcript_text)} characters')
print(f'\nEntities Detected by Category:')
print('-'*40)

total = 0
for category, items in sorted(grouped.items()):
    icon = CATEGORY_ICONS.get(category, '📋')
    print(f'  {icon} {category:<40} {len(items):>3} entities')
    total += len(items)

print('-'*40)
print(f'  Total                                      {total:>3} entities')
print()
print(f'  💊 Medications (RxNorm) : {len(rx_entities)}')
print(f'  🧬 Diagnoses (ICD-10)   : {len(icd_entities)}')
print()
print('S3 Resources Created:')
print(f'  Audio Input    : s3://{BUCKET_AUDIO_INPUT}/')
print(f'  Transcribe Out : s3://{BUCKET_TRANSCRIBE_OUT}/')
print(f'  Comprehend In  : s3://{BUCKET_COMPREHEND_IN}/')
print()
print('✅ Lab Complete!')

if FALLBACK_USED:
    print()
    print('📌 To get real Transcribe output, fix audio by running:')
    print('   ffmpeg -i your_file.mp4 -ar 16000 -ac 1 -acodec pcm_s16le fixed.wav')
    print('   Then upload fixed.wav and restart from Step 3.')

## 🧹 Step 10 — Cleanup (Uncomment to Run)

In [ ]:
import boto3

s3         = boto3.client('s3')
transcribe = boto3.client('transcribe')

# ── Delete S3 Buckets ─────────────────────────────────────────────────────
def empty_and_delete(bucket):
    try:
        paginator = s3.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket):
            objs = [{'Key': o['Key']} for o in page.get('Contents', [])]
            if objs:
                s3.delete_objects(Bucket=bucket, Delete={'Objects': objs})
        s3.delete_bucket(Bucket=bucket)
        print(f'✅ Deleted bucket : {bucket}')
    except Exception as e:
        print(f'⚠️  {bucket} : {e}')

for b in BUCKETS:
    empty_and_delete(b)

# ── Delete Transcribe Job ─────────────────────────────────────────────────
try:
    transcribe.delete_medical_transcription_job(
        MedicalTranscriptionJobName=JOB_NAME
    )
    print(f'✅ Deleted Transcribe job : {JOB_NAME}')
except Exception as e:
    print(f'⚠️  Transcribe job : {e}')

print('\n✅ All resources deleted!')